# Lab 5 — RAG con Ollama (normativa estructural) — Vía IA-asistida

**Sesión:** modelos de lenguaje locales con recuperación aumentada (RAG)

**Público:** ingenieros civiles **sin experiencia en programación** · **Entorno:** GitHub Codespaces o `labs/.venv`

## Cómo trabajar (Copilot, Gemini, Cursor, etc.)

1. Abre este repo en **Codespaces** o activa `source labs/.venv/bin/activate`.
2. **Ejecuta** las celdas pre-escritas (carga de datos, gráficos base) en orden.
3. Lee la **guía IA** de la sección: objetivo, qué considerar y el prompt sugerido.
4. Copia el prompt a tu asistente; pega el código generado en la celda **«Aquí coloca tu solución»**.
5. Ejecuta tu celda y la **Autoevaluación**; busca ✅ antes de avanzar.
6. Registra tus prompts en [`prompts_entregados.md`](prompts_entregados.md) (entrega obligatoria).
7. Referencia docente: `llm_local_estructuras_solucion.ipynb` (no distribuir al inicio).

**La IA propone; tú validas** con autoevaluación, gráficos y sentido físico/estructural.

Corpus: [`data/DATOS.md`](data/DATOS.md) — normas E.020, E.030, E.050 en `pdfs/`.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_pasos_rag, verificar_ollama, verificar_inventario_pdfs,
    verificar_extraccion, verificar_chunks, verificar_embeddings,
    verificar_recuperacion, verificar_rag_respuesta, verificar_comparacion,
    listar_pdfs, extraer_texto_pdf, fragmentar_texto, cosine_top_k, llamar_ollama,
    resumen_seccion, E030_MAX_PAGINAS,
)

import numpy as np
import requests
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer

MODELO_EMB = 'all-MiniLM-L6-v2'
RUTA_PDFS = Path('pdfs')

print('✅ Entorno listo (Codespaces / labs/.venv).')
print('   Embeddings:', MODELO_EMB, '| Ollama: http://localhost:11434')


## Corpus — Normas estructurales peruanas

| PDF | Norma | Uso en obra |
|-----|-------|-------------|
| `Norma_E_020_CARGAS.pdf` | **E.020** | Cargas muertas, vivas, viento |
| `Norma_E_030_SISMO.pdf` | **E.030** | Diseño sismorresistente (muestra 30 pág.) |
| `Norma_E_050_SUELOS.pdf` | **E.050** | Estudios geotécnicos |

Pipeline: **extracción → chunks → embeddings (MiniLM) → top-k → Ollama**.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Contexto RAG en consultas de normativa

| Enfoque | Ventaja | Riesgo |
|---------|---------|--------|
| **Prompt vacío** | Rápido | Alucinaciones, cifras inventadas |
| **RAG local** | Respuesta anclada a PDFs de obra | Calidad del chunk y del modelo pequeño |

### 📘 Subpreguntas
- **1.a** ¿Por qué un ingeniero preferiría RAG local frente a ChatGPT con el PDF subido?
- **1.b** ¿Qué paso del pipeline evita que el LLM «invente» un artículo?


#### ✍️ Tu respuesta (opcional, 2–3 líneas)




### 🤖 Guía IA — Sección 1: Contexto RAG

**Objetivo:** Listar los 5 pasos del pipeline RAG que construirás en este lab.

**Tu código debe lograr**
- Definir `PASOS_RAG` como lista con al menos 5 pasos
- Incluir carga, fragmentación, embeddings, recuperación y generación
- Imprimir la lista numerada

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `PASOS_RAG`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- RAG = recuperar fragmentos de normativa antes de preguntar al LLM
- En obra: consultas sobre E.020/E.030/E.050 sin enviar PDFs a la nube
- Nombres sugeridos: carga PDFs, fragmentación, embeddings, recuperación top-k, generación Ollama

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Estoy en el Lab 5 de RAG con Ollama para normas estructurales peruanas.
Genera código que:
1) defina PASOS_RAG = ["Carga de PDFs", "Fragmentación en chunks", "Embeddings con MiniLM", "Recuperación top-k", "Generación con Ollama"]
2) imprima "Pipeline RAG:"
3) enumere cada paso con print(f"  {i}. {paso}")
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · PASOS_RAG

# Tu código debe:
#   1. Lista con 5 pasos del pipeline RAG
#   2. Imprimir cada paso numerado



In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `PASOS_RAG`
r = []
try:
    r = verificar_pasos_rag(PASOS_RAG)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Contexto RAG', r)


## Pregunta 2 — Verificar Ollama (LLM local)

### 📘 Subpreguntas
- **2.a** ¿Qué ventaja tiene ejecutar el LLM en localhost:11434?
- **2.b** ¿Qué hacer si `OLLAMA_OK` es False?


In [ ]:
# --- PRE-ESCRITO: diagnóstico Ollama ---
# Si falla, ejecuta en terminal: bash labs/lab5/_ollama_setup.sh
print("Comprueba que Ollama esté activo antes de las secciones 8–9.")


### 🤖 Guía IA — Sección 2: Verificar Ollama

**Objetivo:** Configurar el modelo LLM local y comprobar que Ollama responde.

**Tu código debe lograr**
- Definir `MODELO_LLM = 'llama3.2:3b'` y `OLLAMA_URL = 'http://localhost:11434'`
- Hacer GET a `/api/tags` con requests
- Asignar `OLLAMA_OK = True` si responde 200, si no False
- Imprimir estado y modelos disponibles

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `MODELO_LLM`, `OLLAMA_OK`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Si OLLAMA_OK es False ejecuta en terminal: bash labs/lab5/_ollama_setup.sh
- La primera descarga del modelo puede tardar varios minutos
- No uses APIs en la nube — solo localhost:11434

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter del Lab 5 RAG necesito verificar Ollama local.
Genera código que:
1) defina MODELO_LLM = "llama3.2:3b"
2) defina OLLAMA_URL = "http://localhost:11434"
3) intente requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
4) asigne OLLAMA_OK = (resp.status_code == 200) en try/except: OLLAMA_OK = False
5) imprima OLLAMA_OK y, si True, los nombres de modelos del JSON
Usa import requests (ya disponible).
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · MODELO_LLM
#   · OLLAMA_URL
#   · OLLAMA_OK

# Tu código debe:
#   1. Definir MODELO_LLM y OLLAMA_URL
#   2. Comprobar conexión con /api/tags
#   3. Asignar OLLAMA_OK e imprimir diagnóstico



In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `MODELO_LLM`, `OLLAMA_OK`
r = []
try:
    r = verificar_ollama(MODELO_LLM, OLLAMA_OK)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Ollama', r)


## Pregunta 3 — Inventario de PDFs

### 📘 Subpreguntas
- **3.a** ¿Cuántos documentos normativos hay en `pdfs/`?
- **3.b** ¿Por qué empezamos con E.020 (cargas)?


In [ ]:
# --- PRE-ESCRITO: listar corpus ---
lista_pdfs = listar_pdfs(RUTA_PDFS)
for p in lista_pdfs:
    print(f"  · {p.name} ({p.stat().st_size // 1024} KB)")


### 🤖 Guía IA — Sección 3: Inventario de PDFs

**Objetivo:** Elegir el PDF activo para empezar la extracción (E.020 cargas).

**Tu código debe lograr**
- Definir `N_PDFS` = número de archivos en pdfs/
- Definir `PDF_ACTIVO = 'Norma_E_020_CARGAS.pdf'`
- Imprimir ambos valores

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `PDF_ACTIVO`, `N_PDFS`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- La celda pre-escrita ya listó los PDFs con listar_pdfs()
- Debe haber exactamente 3 PDFs: E.020, E.030, E.050
- Empezamos con E.020 (cargas) por ser el más manejable

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter la celda anterior definió lista_pdfs = listar_pdfs("pdfs").
Genera código que:
1) defina N_PDFS = len(lista_pdfs)
2) defina PDF_ACTIVO = "Norma_E_020_CARGAS.pdf"
3) imprima N_PDFS y PDF_ACTIVO
No recalcules lista_pdfs.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · PDF_ACTIVO
#   · N_PDFS

# Tu código debe:
#   1. N_PDFS = len(lista_pdfs)
#   2. PDF_ACTIVO = Norma_E_020_CARGAS.pdf
#   3. Imprimir inventario



In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `PDF_ACTIVO`, `N_PDFS`
r = []
try:
    r = verificar_inventario_pdfs(PDF_ACTIVO, N_PDFS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Inventario PDFs', r)


## Pregunta 4 — Extracción de texto (pypdf)

### 📘 Subpreguntas
- **4.a** ¿Por qué extraemos texto antes de preguntar al LLM?
- **4.b** ¿Qué limitación tienen los PDFs escaneados (solo imagen)?


In [ ]:
# --- PRE-ESCRITO: ruta del PDF activo ---
ruta_activo = RUTA_PDFS / PDF_ACTIVO
print(f"Extrayendo: {ruta_activo}")


### 🤖 Guía IA — Sección 4: Extracción de texto

**Objetivo:** Extraer texto del PDF activo con pypdf.

**Tu código debe lograr**
- Usar `extraer_texto_pdf(Path('pdfs') / PDF_ACTIVO)`
- Guardar en `texto_pdf`, `N_PAGINAS` y `N_CARACTERES`
- Mostrar los primeros 400 caracteres

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `texto_pdf`, `N_PAGINAS`, `N_CARACTERES`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- extraer_texto_pdf ya está importada desde _verificar
- N_CARACTERES = len(texto_pdf)
- Si el texto es muy corto, el PDF puede ser escaneado (fuera de alcance del lab)

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo PDF_ACTIVO y la función extraer_texto_pdf(ruta) -> (texto, n_paginas_total).
Genera código que:
1) from pathlib import Path
2) texto_pdf, N_PAGINAS = extraer_texto_pdf(Path("pdfs") / PDF_ACTIVO)
3) N_CARACTERES = len(texto_pdf)
4) imprima N_PAGINAS, N_CARACTERES
5) print(texto_pdf[:400])
No uses imports nuevos salvo Path si hace falta.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · texto_pdf
#   · N_PAGINAS
#   · N_CARACTERES

# Tu código debe:
#   1. Extraer texto del PDF activo
#   2. Calcular N_CARACTERES
#   3. Mostrar muestra de 400 caracteres



In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `texto_pdf`, `N_PAGINAS`, `N_CARACTERES`
r = []
try:
    r = verificar_extraccion(texto_pdf, N_PAGINAS, N_CARACTERES)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Extracción', r)


## Pregunta 5 — Fragmentación en chunks

### 📘 Subpreguntas
- **5.a** ¿Por qué no enviamos el PDF completo al LLM?
- **5.b** ¿Para qué sirve `CHUNK_OVERLAP`?


In [ ]:
# --- PRE-ESCRITO: parámetros sugeridos ---
# CHUNK_SIZE ≈ 600–1000 caracteres | CHUNK_OVERLAP ≈ 10–15 % del tamaño
print("Ajusta CHUNK_SIZE si obtienes demasiados o muy pocos fragmentos.")


### 🤖 Guía IA — Sección 5: Fragmentación

**Objetivo:** Dividir el texto en chunks solapados para la búsqueda.

**Tu código debe lograr**
- Definir `CHUNK_SIZE = 800` y `CHUNK_OVERLAP = 100`
- Crear `CHUNKS = fragmentar_texto(texto_pdf, CHUNK_SIZE, CHUNK_OVERLAP)`
- Definir `N_CHUNKS = len(CHUNKS)` e imprimir

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `CHUNKS`, `N_CHUNKS`, `CHUNK_SIZE`, `CHUNK_OVERLAP`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- fragmentar_texto está en _verificar
- Chunks más pequeños = más precisión pero más vectores
- CHUNK_OVERLAP evita cortar artículos de norma a la mitad

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo texto_pdf y fragmentar_texto(texto, chunk_size, chunk_overlap).
Genera código que:
1) CHUNK_SIZE = 800; CHUNK_OVERLAP = 100
2) CHUNKS = fragmentar_texto(texto_pdf, CHUNK_SIZE, CHUNK_OVERLAP)
3) N_CHUNKS = len(CHUNKS)
4) imprima N_CHUNKS y el primer chunk ([:200])
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · CHUNKS
#   · N_CHUNKS
#   · CHUNK_SIZE
#   · CHUNK_OVERLAP

# Tu código debe:
#   1. Definir CHUNK_SIZE y CHUNK_OVERLAP
#   2. Crear CHUNKS con fragmentar_texto
#   3. Imprimir N_CHUNKS



In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `CHUNKS`, `N_CHUNKS`, `CHUNK_SIZE`, `CHUNK_OVERLAP`
r = []
try:
    r = verificar_chunks(CHUNKS, N_CHUNKS, CHUNK_SIZE, CHUNK_OVERLAP)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — Fragmentación', r)


## Pregunta 6 — Embeddings e índice en memoria

### 📘 Subpreguntas
- **6.a** ¿Qué representa cada fila de `EMBEDDINGS`?
- **6.b** ¿Por qué usamos MiniLM y reservamos Ollama solo para generación?


In [ ]:
# --- PRE-ESCRITO: indexar E.050 y muestra E.030 ---
def _texto_norma(nombre: str, max_pag: int | None = None) -> str:
    t, _ = extraer_texto_pdf(RUTA_PDFS / nombre, max_paginas=max_pag)
    return t

texto_e050, _ = extraer_texto_pdf(RUTA_PDFS / 'Norma_E_050_SUELOS.pdf')
texto_e030, _ = extraer_texto_pdf(RUTA_PDFS / 'Norma_E_030_SISMO.pdf', max_paginas=E030_MAX_PAGINAS)
texto_corpus = texto_pdf + '\n' + texto_e050 + '\n' + texto_e030
CHUNKS_CORPUS = fragmentar_texto(texto_corpus, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Corpus ampliado: {len(CHUNKS_CORPUS)} chunks (E.020 + E.050 + E.030 parcial)")


### 🤖 Guía IA — Sección 6: Embeddings e índice

**Objetivo:** Vectorizar chunks con sentence-transformers (all-MiniLM-L6-v2).

**Tu código debe lograr**
- Cargar `modelo_emb = SentenceTransformer(MODELO_EMB)`
- Calcular `EMBEDDINGS = modelo_emb.encode(CHUNKS, show_progress_bar=False)`
- Definir `N_VECTORES = EMBEDDINGS.shape[0]` e imprimir forma

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `EMBEDDINGS`, `N_VECTORES`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- MODELO_EMB ya está definido en la celda setup: 'all-MiniLM-L6-v2'
- La primera ejecución descarga el modelo (~90 MB)
- N_VECTORES debe igualar N_CHUNKS

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo CHUNKS, N_CHUNKS, MODELO_EMB = "all-MiniLM-L6-v2" y SentenceTransformer importado.
Genera código que:
1) modelo_emb = SentenceTransformer(MODELO_EMB)
2) EMBEDDINGS = modelo_emb.encode(CHUNKS, show_progress_bar=False)
3) N_VECTORES = EMBEDDINGS.shape[0]
4) imprima N_VECTORES y EMBEDDINGS.shape
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · EMBEDDINGS
#   · N_VECTORES

# Tu código debe:
#   1. Instanciar SentenceTransformer
#   2. Codificar CHUNKS en EMBEDDINGS
#   3. Imprimir N_VECTORES y shape



In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `EMBEDDINGS`, `N_VECTORES`
r = []
try:
    r = verificar_embeddings(EMBEDDINGS, N_VECTORES, N_CHUNKS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Embeddings', r)


## Pregunta 7 — Recuperación top-k

### 📘 Subpreguntas
- **7.a** ¿Qué mide la similitud coseno entre consulta y chunk?
- **7.b** ¿Qué pasa si TOP_K es demasiado grande?


In [ ]:
# --- PRE-ESCRITO: visualizar función de recuperación ---
display(Markdown(
    "**Recuperación:** embed(query) → similitud coseno con cada fila de EMBEDDINGS → top-k chunks."
))


### 🤖 Guía IA — Sección 7: Recuperación top-k

**Objetivo:** Buscar los fragmentos más similares a una consulta técnica.

**Tu código debe lograr**
- Definir `CONSULTA` sobre cargas o norma E.020
- Definir `TOP_K = 3`
- Codificar consulta, usar `cosine_top_k`, llenar `CHUNKS_RECUPERADOS`

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `CONSULTA`, `CHUNKS_RECUPERADOS`, `TOP_K`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- cosine_top_k está en _verificar
- Muestra los fragmentos recuperados antes de la sección 8
- Ejemplo: '¿Qué tipos de sobrecarga considera la norma E.020?'

**Prompt listo para copiar** (sección avanzada — úsalo tal cual y adapta solo si la IA falla):

```text
En Jupyter tengo CHUNKS, EMBEDDINGS, modelo_emb, cosine_top_k(query_vec, matrix, k).
Genera código que:
1) CONSULTA = "¿Qué tipos de sobrecarga considera la norma E.020 de cargas?"
2) TOP_K = 3
3) q_vec = modelo_emb.encode([CONSULTA])[0]
4) idx, scores = cosine_top_k(q_vec, EMBEDDINGS, TOP_K)
5) CHUNKS_RECUPERADOS = [CHUNKS[i] for i in idx]
6) imprima CONSULTA y cada fragmento con su score
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · CONSULTA
#   · TOP_K
#   · CHUNKS_RECUPERADOS

# Tu código debe:
#   1. Definir CONSULTA técnica
#   2. TOP_K = 3 y cosine_top_k
#   3. Llenar CHUNKS_RECUPERADOS



In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `CONSULTA`, `CHUNKS_RECUPERADOS`, `TOP_K`
r = []
try:
    r = verificar_recuperacion(CONSULTA, CHUNKS_RECUPERADOS, TOP_K)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Recuperación', r)


## Pregunta 8 — Prompt RAG y respuesta con Ollama

### 📘 Subpreguntas
- **8.a** ¿Qué instrucción reduce alucinaciones en el prompt?
- **8.b** ¿Por qué mostramos los fragmentos recuperados antes de la respuesta?


In [ ]:
# --- PRE-ESCRITO: plantilla de system prompt ---
PLANTILLA_SISTEMA = """Responde SOLO con base en el CONTEXTO.
Si la respuesta no está en el contexto, di "No consta en el contexto".
Cita artículo o sección cuando sea posible."""
print(PLANTILLA_SISTEMA)


### 🤖 Guía IA — Sección 8: Prompt RAG y respuesta Ollama

**Objetivo:** Armar prompt con contexto recuperado y generar respuesta local.

**Tu código debe lograr**
- Unir CHUNKS_RECUPERADOS en `contexto`
- Armar `PROMPT_RAG` con instrucciones + contexto + CONSULTA
- Llamar `llamar_ollama(PROMPT_RAG, MODELO_LLM, OLLAMA_URL)` → `RESPUESTA_RAG`

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `PROMPT_RAG`, `RESPUESTA_RAG`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Requiere OLLAMA_OK = True (sección 2)
- Pide al modelo citar solo el contexto; si no sabe, decir 'no consta'
- llamar_ollama está en _verificar

**Prompt listo para copiar** (sección avanzada — úsalo tal cual y adapta solo si la IA falla):

```text
En Jupyter tengo CONSULTA, CHUNKS_RECUPERADOS, MODELO_LLM, OLLAMA_URL y llamar_ollama().
Genera código que:
1) contexto = "\n---\n".join(CHUNKS_RECUPERADOS)
2) PROMPT_RAG = f'''Eres un asistente de normativa estructural peruana.
Responde SOLO con base en el CONTEXTO. Si no está, di "No consta en el contexto".

CONTEXTO:
{contexto}

PREGUNTA: {CONSULTA}
'''
3) RESPUESTA_RAG = llamar_ollama(PROMPT_RAG, MODELO_LLM, OLLAMA_URL)
4) print(PROMPT_RAG[:300], "...")
5) print("RESPUESTA:", RESPUESTA_RAG)
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · PROMPT_RAG
#   · RESPUESTA_RAG

# Tu código debe:
#   1. Armar PROMPT_RAG con contexto
#   2. Llamar Ollama
#   3. Imprimir respuesta

# Nota: Requiere Ollama activo (sección 2).



In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `PROMPT_RAG`, `RESPUESTA_RAG`
r = []
try:
    r = verificar_rag_respuesta(PROMPT_RAG, RESPUESTA_RAG)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Prompt RAG', r)


## Pregunta 9 — RAG vs consulta sin contexto

### 📘 Subpreguntas
- **9.a** ¿Qué diferencias observas entre ambas respuestas?
- **9.b** ¿Cuándo **no** deberías confiar en la salida del LLM?


In [ ]:
# --- PRE-ESCRITO: misma consulta, sin chunks ---
print(f"Consulta base: {CONSULTA}")
print("Generaremos RESPUESTA_SIN_RAG sin contexto recuperado.")


### 🤖 Guía IA — Sección 9: RAG vs sin contexto

**Objetivo:** Comparar respuesta con y sin fragmentos recuperados; detectar uso de fuente.

**Tu código debe lograr**
- Generar `RESPUESTA_SIN_RAG` con la misma CONSULTA pero sin contexto
- Comparar longitud o contenido de ambas respuestas
- Definir `USA_FUENTE = True` si RESPUESTA_RAG menciona norma, E.020, artículo o texto del contexto

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `RESPUESTA_SIN_RAG`, `USA_FUENTE`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Sin contexto el modelo suele ser más genérico o alucinar
- USA_FUENTE es True solo si hay evidencia de uso del contexto
- Imprime ambas respuestas lado a lado para comparar

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo CONSULTA, RESPUESTA_RAG, MODELO_LLM, OLLAMA_URL, llamar_ollama().
Genera código que:
1) prompt_vacio = f"Responde brevemente: {CONSULTA}"
2) RESPUESTA_SIN_RAG = llamar_ollama(prompt_vacio, MODELO_LLM, OLLAMA_URL)
3) imprima "CON RAG:", RESPUESTA_RAG[:500]
4) imprima "SIN RAG:", RESPUESTA_SIN_RAG[:500]
5) USA_FUENTE = any(p in RESPUESTA_RAG.lower() for p in ["e.020", "norma", "artículo", "sobrecarga", "carga"])
6) print("USA_FUENTE:", USA_FUENTE)
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · RESPUESTA_SIN_RAG
#   · USA_FUENTE

# Tu código debe:
#   1. Generar RESPUESTA_SIN_RAG sin contexto
#   2. Comparar con RESPUESTA_RAG
#   3. Evaluar USA_FUENTE



In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `RESPUESTA_SIN_RAG`, `USA_FUENTE`
r = []
try:
    r = verificar_comparacion(RESPUESTA_SIN_RAG, USA_FUENTE, RESPUESTA_RAG)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — Comparación RAG', r)


## Cierre y entrega (vía IA)

### ✍️ Reflexión final (3 frases)
- RAG ayudó a ___ porque ___.
- Sin contexto el modelo tendió a ___.
- Antes de usar esto en obra real verificaría ___.

### 📋 Bitácora de prompts (obligatorio)

Completa [`prompts_entregados.md`](prompts_entregados.md): por cada sección, copia el prompt enviado, un resumen de la respuesta de la IA y qué aceptaste o rechazaste.

---
**Checklist:** 9 autoevaluaciones en ✅ · comparaste RAG vs sin contexto · bitácora entregada.
